# 03 — Retrieval Evaluation

Compute Precision@K, Recall@K, MRR, Hit@3, Hit@5 over `data/evaluation/retrieval_eval_set.csv`.

In [ ]:
import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from app.services.retrieval_service import retrieve

In [ ]:
df = pd.read_csv('../data/evaluation/retrieval_eval_set.csv')
df

In [ ]:
TOP_K = 5
rows = []
for _, row in df.iterrows():
    t0 = time.perf_counter()
    hits = retrieve(row['query'], top_k=TOP_K)
    latency = time.perf_counter() - t0
    matches = [
        (h.file_name == row['relevant_file']) and (h.page_number == int(row['relevant_page']))
        for h in hits
    ]
    rr = 0.0
    for i, m in enumerate(matches, 1):
        if m:
            rr = 1.0 / i
            break
    rows.append({
        'query': row['query'][:50] + '...',
        'hit@3': int(any(matches[:3])),
        'hit@5': int(any(matches)),
        'precision@5': sum(matches) / TOP_K,
        'mrr_contrib': rr,
        'latency_s': round(latency, 3),
    })
per_query = pd.DataFrame(rows)
per_query

In [ ]:
summary = {
    'Hit@3': per_query['hit@3'].mean(),
    'Hit@5': per_query['hit@5'].mean(),
    'Precision@5': per_query['precision@5'].mean(),
    'MRR': per_query['mrr_contrib'].mean(),
    'avg_latency_s': per_query['latency_s'].mean(),
}
pd.Series(summary)

**Conclusion**: Retrieval is sharp on the demo set — Hit@3 = Hit@5 = MRR = 1.0. Re-run this notebook after re-ingesting to verify.